# Hướng dẫn chạy notebook TV2

## 1. Dữ liệu đầu vào

Thêm dataset mitbih-tv1-v2 vào mục Input của notebook.

Dataset cần có ba tệp:
- ds1.npz
- ds2.npz
- preprocess_config.json

Đường dẫn dữ liệu:
 /kaggle/input/datasets/quanghuy10/mitbih-tv1-v2

Các trường trong mỗi tệp NPZ:
X_waveform, X_rr, y, record_id, patient_id, r_peak_index.

ECG có kích thước [N, 1, 234].
RR có kích thước [N, 3], theo thứ tự:
RR_pre, RR_post, RR_local_avg; đơn vị giây.

Thứ tự lớp: N, S, V, F, Q.

## 2. Môi trường chạy

Notebook chạy trên Kaggle với GPU.
Lần chạy gốc sử dụng Tesla T4 và PyTorch 2.11.0+cu128.

Cell cài đặt thư viện sử dụng:
onnx==1.22.0
onnxruntime==1.30.0
onnxscript==0.7.2

Bật Internet khi cài thư viện.
ONNX Runtime sử dụng CPUExecutionProvider để kiểm tra mô hình xuất.

## 3. Chạy lại huấn luyện và đánh giá

Chạy lần lượt 13 cell code đầu tiên, từ trên xuống dưới:

1. Cài thư viện.
2. Khai báo cấu hình và các hàm dùng chung.
3. Đọc và kiểm tra DS1 cùng cấu hình tiền xử lý.
4. Khai báo mô hình và các hàm huấn luyện.
5. Chạy GroupKFold và tìm cấu hình trên DS1.
6. Tổng hợp kết quả DS1 và ablation.
7. Huấn luyện mô hình cuối trên toàn DS1.
8. Đánh giá trên DS2.
9. Xuất các trường hợp sai và ảnh saliency.
10. Xuất ONNX FP32 và tạo phiên bản INT8.
11. Đối chiếu PyTorch, ONNX FP32 và ONNX INT8 trên DS2.
12. Kiểm tra và đóng gói kết quả.
13. Hiển thị liên kết tải ZIP.

Có hai mô hình: baseline và cnn_rr.
Ba seed huấn luyện: 42, 123, 2026.
GroupKFold gồm 5 fold theo record_id trong DS1.
Hai cấu hình learning rate tạo tổng cộng 60 lượt CV.
Mỗi lượt tối đa 30 epoch, early stopping patience bằng 7.

DS2 được sử dụng để đánh giá sau khi chốt cấu hình từ DS1.
Mô hình triển khai là cnn_rr, seed 42, ONNX FP32.

## 4. Vị trí kết quả

Mỗi lần chạy tạo một thư mục:
 /kaggle/working/TV2_MITBIH_<RUN_ID>

Các thư mục bên trong:
- checkpoints: checkpoint PyTorch.
- logs: lịch sử huấn luyện.
- results: bảng đánh giá và dự đoán.
- exports: mô hình ONNX.
- saliency: ảnh và dữ liệu minh họa.

Dự đoán DS2 nằm tại:
 results/ds2_predictions.csv

Thông tin các trường hợp saliency nằm tại:
 results/saliency_cases.csv

Cell đóng gói tạo ZIP; cell tiếp theo cung cấp liên kết tải.

## 5. Tạo lại hình báo cáo từ kết quả đã lưu

Các cell code từ 14 đến 22 dùng để khôi phục kết quả
và tạo hình báo cáo, không cần huấn luyện lại.

Thêm dataset tv2-results vào Input.
Đường dẫn được dùng:
 /kaggle/input/datasets/quanghuy10/tv2-results

Chạy lần lượt các cell khôi phục và tạo hình.
Kết quả được khôi phục tại:
 /kaggle/working/TV2_RESTORED

Hình báo cáo nằm trong:
 figures_report

Cell cuối đóng gói hình và các minh chứng thành ZIP.

# Danh sách minh chứng TV2

## Mô hình và huấn luyện

Cell code 2: cấu hình thí nghiệm, ba seed 42, 123, 2026,
hai learning rate và ngân sách huấn luyện.

Cell code 4: kiến trúc CNN cơ sở và CNN + RR,
các hàm chuẩn hóa, huấn luyện và đánh giá.

Cell code 5: GroupKFold theo bản ghi trong DS1,
chuẩn hóa từng fold và tìm cấu hình.

Cell code 6: tổng hợp kết quả DS1 và ablation.

Cell code 7: huấn luyện mô hình cuối trên toàn DS1.

## Đánh giá trên DS2

Cell code 8: kiểm tra giao bản ghi và bệnh nhân giữa DS1/DS2,
đánh giá hai mô hình trên ba seed và bảng ablation.

results/ds2_predictions.csv:
Dự đoán trên DS2 của từng mô hình và seed.

Kết quả CNN + RR trung bình ba seed:
F1 macro = 0,4875 ± 0,0052.

Kết quả CNN + RR seed 42:
F1 macro = 0,488642.
Accuracy = 0,855521.

Trung bình ± độ lệch chuẩn của ba seed được trình bày
riêng với kết quả mô hình triển khai seed 42.

## Phân tích nhịp sai

Cell code 9: chọn 10 nhịp sai và 2 nhịp đúng,
tính gradient và xuất hình saliency.

results/saliency_cases.csv:
Thông tin, nhãn, xác suất và đặc trưng RR của các nhịp minh họa.

saliency/sample_<sample_index>.png:
Ảnh ECG và saliency của từng trường hợp.

Tài liệu phân tích bổ sung:
TV2_Phan_tich_10_nhip_du_doan_sai.docx.

## Chuyển đổi và lượng tử hóa

Cell code 10: xuất ONNX FP32 và tạo ONNX INT8.
Cell code 11: đối chiếu trên toàn DS2.

ONNX FP32 khớp 100% nhãn với PyTorch.
ONNX INT8 có 2.628 nhãn thay đổi so với PyTorch.

F1 macro:
PyTorch FP32 = 0,488642.
ONNX FP32 = 0,488642.
ONNX INT8 = 0,472451.

Mô hình triển khai: CNN + RR, seed 42, ONNX FP32.

## Hình báo cáo

Cell code 16: đường cong huấn luyện.
Cell code 17: ma trận nhầm lẫn.
Cell code 18: so sánh các chỉ số DS2.
Cell code 19: ablation nhánh RR.
Cell code 20: so sánh FP32 và INT8.
Cell code 21: sơ đồ kiến trúc hai mô hình.
Cell code 22: đóng gói hình báo cáo.

Các hình nằm trong figures_report của TV2_REPORT_FIGURES.zip.

## Khoảng tin cậy bootstrap

TV1 phụ trách gửi code và bảng CI 95% cuối cho
đủ sáu cặp mô hình–seed, lấy mẫu theo bản ghi 1.000 lần.

Bảng bootstrap sẽ được đối chiếu với kết quả DS2
trước khi đưa vào báo cáo.

In [ ]:
%pip install -q onnx==1.22.0 onnxruntime==1.30.0 onnxscript==0.7.2

In [ ]:
import os

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys
import json
import math
import random
import shutil
import hashlib
import platform
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

import onnx
import onnxruntime as ort

from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process

DATA_DIR = Path("/kaggle/input/datasets/quanghuy10/mitbih-tv1")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("/kaggle/working") / f"TV2_MITBIH_{RUN_ID}"

for folder in ["checkpoints", "logs", "results", "exports", "saliency"]:
    (OUT_DIR / folder).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = ["N", "S", "V", "F", "Q"]
RR_ORDER = ["RR_pre", "RR_post", "RR_local_avg"]

SEEDS = [42, 123, 2026]
N_FOLDS = 5
MAX_EPOCHS = 30
PATIENCE = 7

CONFIGS = [
    {"config_id": "lr_001", "learning_rate": 0.001, "batch_size": 128},
    {"config_id": "lr_0003", "learning_rate": 0.0003, "batch_size": 128},
]

DS1_RECORDS = [
    "101", "106", "108", "109", "112", "114", "115", "116",
    "118", "119", "122", "124", "201", "203", "205", "207",
    "208", "209", "215", "220", "223", "230",
]

DS2_RECORDS = [
    "100", "103", "105", "111", "113", "117", "121", "123",
    "200", "210", "212", "213", "214", "219", "221", "222",
    "228", "231", "232", "233", "234",
]

EXPORT_SEED = 42
DEPLOYMENT_VARIANT = "cnn_rr"
DEPLOYMENT_FORMAT = "onnx_fp32"

METRICS = [
    "se_S", "ppv_S", "se_V", "ppv_V",
    "f1_macro", "sp_macro", "accuracy",
]

def save_json(data, path):
    Path(path).write_text(
        json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )

def save_csv(data, path):
    frame = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    frame.to_csv(path, index=False, encoding="utf-8-sig", na_rep="")
    return frame

def file_hash(path, algorithm="sha256"):
    digest = hashlib.new(algorithm)
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)

protocol = {
    "dataset": "MIT-BIH Arrhythmia Database",
    "classes": CLASS_NAMES,
    "rr_order": RR_ORDER,
    "seeds": SEEDS,
    "folds": N_FOLDS,
    "configs": CONFIGS,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "optimizer": "Adam",
    "loss": "weighted_cross_entropy",
    "checkpoint_metric": "validation_f1_macro",
    "checkpoint_tie_rule": "keep_earliest_epoch",
    "configuration_selection": "highest_mean_cv_f1_across_both_models_and_seeds",
    "final_epoch_rule": "ceil_median_best_epoch_of_selected_configuration",
    "decision_rule": "argmax",
    "hypothesis": (
        "Bổ sung RR có thể giúp phân biệt các nhịp có hình dạng "
        "gần giống nhau nhưng khác thời điểm xuất hiện."
    ),
    "deployment_variant": DEPLOYMENT_VARIANT,
    "deployment_seed": EXPORT_SEED,
    "deployment_format": DEPLOYMENT_FORMAT,
    "int8_role": "quantization_comparison",
    "excluded_paced_records": ["102", "104", "107", "217"],
    "excluded_test_record": "202",
}

save_json(protocol, OUT_DIR / "protocol.json")

print("Thiết bị:", DEVICE)
print("PyTorch:", torch.__version__)
print("Thư mục kết quả:", OUT_DIR)
print("Số lượt CV:", len(CONFIGS) * 2 * len(SEEDS) * N_FOLDS)

In [ ]:
DATA_DIR = Path("/kaggle/input/datasets/quanghuy10/mitbih-tv1-v2")

config_path = DATA_DIR / "preprocess_config.json"

if not config_path.is_file():
    raise FileNotFoundError(
        f"Chưa tìm thấy {config_path}. Hãy cập nhật dataset trong Input."
    )

with config_path.open(encoding="utf-8-sig") as file:
    source_config = json.load(file)

assert source_config["sampling_rate_hz"] == 360
assert source_config["beat_window"]["left_samples"] == 90
assert source_config["beat_window"]["right_samples"] == 144
assert source_config["beat_window"]["total_samples"] == 234
assert source_config["beat_window"]["ecg_tensor_shape"] == "[N, 1, 234]"
assert source_config["rr_features"]["rr_tensor_shape"] == "[N, 3]"
assert source_config["rr_features"]["vector_format"] == RR_ORDER
assert source_config["class_mapping"] == {
    "N": 0, "S": 1, "V": 2, "F": 3, "Q": 4
}
assert source_config["r_peak_indexing"].startswith("0-based")
assert source_config["normalization"].startswith("none")
assert source_config["rr_features"]["unit"].startswith("seconds")
assert source_config["filtering"]["lowcut_hz"] == 0.5
assert source_config["filtering"]["highcut_hz"] == 40.0
assert source_config["filtering"]["zero_phase"] is True

preprocess_config = {
    "fs": source_config["sampling_rate_hz"],
    "window_before": source_config["beat_window"]["left_samples"],
    "window_after": source_config["beat_window"]["right_samples"],
    "r_index_base": 0,
    "class_names": CLASS_NAMES,
    "rr_order": RR_ORDER,
    "rr_unit": "seconds",
    "normalized": False,
    "bandpass_hz": [
        source_config["filtering"]["lowcut_hz"],
        source_config["filtering"]["highcut_hz"]
    ],
    "channel_rule": source_config["ecg_channel"],
    "filter_description": source_config["filtering"],
    "rr_local_avg_rule": source_config["rr_features"][
        "rr_local_avg_calculation"
    ],
    "boundary_rule": source_config["rr_features"]["edge_beats_handling"],
    "source_config": source_config
}

PREPROCESS_CONFIG = preprocess_config


def read_dataset(path, expected_records):
    required = [
        "X_waveform",
        "X_rr",
        "y",
        "record_id",
        "patient_id",
        "r_peak_index"
    ]

    with np.load(path, allow_pickle=False) as data:
        missing = set(required) - set(data.files)

        if missing:
            raise ValueError(f"{path.name} thiếu trường: {sorted(missing)}")

        arrays = {name: data[name] for name in required}

    labels = arrays["y"]
    r_peaks = arrays["r_peak_index"]
    n = len(labels)

    if n == 0:
        raise ValueError(f"{path.name} không có nhịp.")

    if not np.issubdtype(labels.dtype, np.integer):
        raise ValueError("Nhãn phải có kiểu số nguyên.")

    if not np.issubdtype(r_peaks.dtype, np.integer):
        raise ValueError("Chỉ số đỉnh R phải có kiểu số nguyên.")

    if arrays["X_waveform"].shape != (n, 1, 234):
        raise ValueError("ECG phải có kích thước [N, 1, 234].")

    if arrays["X_rr"].shape != (n, 3):
        raise ValueError("RR phải có kích thước [N, 3].")

    for name in ["y", "record_id", "patient_id", "r_peak_index"]:
        if arrays[name].shape != (n,):
            raise ValueError(f"{name} phải có kích thước [N].")

    result = {
        "ecg": np.ascontiguousarray(
            arrays["X_waveform"], dtype=np.float32
        ),
        "rr": np.ascontiguousarray(
            arrays["X_rr"], dtype=np.float32
        ),
        "labels": labels.astype(np.int64),
        "record_id": np.char.strip(arrays["record_id"].astype(str)),
        "patient_id": np.char.strip(arrays["patient_id"].astype(str)),
        "r_peak_index": r_peaks.astype(np.int64)
    }

    if not np.isfinite(result["ecg"]).all():
        raise ValueError("ECG chứa NaN hoặc Inf.")

    if not np.isfinite(result["rr"]).all():
        raise ValueError("RR chứa NaN hoặc Inf.")

    if np.any(result["rr"] <= 0):
        raise ValueError("RR chứa giá trị không dương.")

    if np.any((result["labels"] < 0) | (result["labels"] > 4)):
        raise ValueError("Nhãn phải nằm trong khoảng 0 đến 4.")

    if np.any(result["r_peak_index"] < 90):
        raise ValueError("Có đỉnh R không đủ 90 mẫu phía trước.")

    if np.any(result["patient_id"] == ""):
        raise ValueError("Có mã bệnh nhân rỗng.")

    observed = set(result["record_id"].tolist())
    expected = {str(record) for record in expected_records}

    if observed != expected:
        raise ValueError(
            f"{path.name}: thiếu bản ghi {sorted(expected - observed)}, "
            f"thừa bản ghi {sorted(observed - expected)}"
        )

    metadata = pd.DataFrame({
        "record_id": result["record_id"],
        "patient_id": result["patient_id"],
        "r_peak_index": result["r_peak_index"]
    })

    if metadata.duplicated(["record_id", "r_peak_index"]).any():
        raise ValueError("Có đỉnh R trùng trong cùng bản ghi.")

    patient_counts = metadata.groupby("record_id")["patient_id"].nunique()

    if np.any(patient_counts.to_numpy() != 1):
        raise ValueError("Một bản ghi đang có nhiều mã bệnh nhân.")

    for record_id, rows in metadata.groupby("record_id", sort=False):
        peaks = rows["r_peak_index"].to_numpy()

        if np.any(np.diff(peaks) <= 0):
            raise ValueError(
                f"Đỉnh R của bản ghi {record_id} chưa được xếp tăng dần."
            )

    return result


ds1 = read_dataset(DATA_DIR / "ds1.npz", DS1_RECORDS)

save_json(preprocess_config, OUT_DIR / "preprocess_config.json")

counts = np.bincount(ds1["labels"], minlength=5)

print("Đường dẫn dữ liệu:", DATA_DIR)
print("Số nhịp DS1:", len(ds1["labels"]))
print("ECG:", ds1["ecg"].shape)
print("RR:", ds1["rr"].shape)
print("Số bản ghi:", len(np.unique(ds1["record_id"])))
print("Dữ liệu đầu vào chưa chuẩn hóa theo cấu hình TV1.")
print("Đơn vị RR theo cấu hình TV1: giây.")
print("Đỉnh R tăng dần, không trùng trong từng bản ghi.")

for name, count in zip(CLASS_NAMES, counts):
    print(f"{name}: {count} nhịp")

print("\nCell 3: đọc và kiểm tra DS1 thành công.")

In [ ]:
class ECGEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(start_dim=1),
        )

    def forward(self, ecg):
        return self.layers(ecg)


class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ECGEncoder()
        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 5),
        )

    def forward(self, ecg, rr=None):
        return self.classifier(self.encoder(ecg))


class CNNWithRR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ECGEncoder()
        self.rr_branch = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(80, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 5),
        )

    def forward(self, ecg, rr):
        ecg_features = self.encoder(ecg)
        rr_features = self.rr_branch(rr)
        features = torch.cat([ecg_features, rr_features], dim=1)
        return self.classifier(features)


MODEL_CLASSES = {
    "baseline": BaselineCNN,
    "cnn_rr": CNNWithRR,
}

def fit_normalization(ecg, rr):
    return {
        "ecg_mean": float(ecg.mean(dtype=np.float64)),
        "ecg_std": max(float(ecg.std(dtype=np.float64)), 1e-8),
        "rr_mean": rr.mean(axis=0, dtype=np.float64).tolist(),
        "rr_std": np.maximum(
            rr.std(axis=0, dtype=np.float64), 1e-8
        ).tolist(),
    }

def normalize(ecg, rr, parameters):
    ecg_scaled = (
        ecg.astype(np.float64) - parameters["ecg_mean"]
    ) / parameters["ecg_std"]

    rr_scaled = (
        rr.astype(np.float64) - np.asarray(parameters["rr_mean"])
    ) / np.asarray(parameters["rr_std"])

    ecg_scaled = np.ascontiguousarray(ecg_scaled, dtype=np.float32)
    rr_scaled = np.ascontiguousarray(rr_scaled, dtype=np.float32)

    if not np.isfinite(ecg_scaled).all() or not np.isfinite(rr_scaled).all():
        raise ValueError("Dữ liệu sau chuẩn hóa không hữu hạn.")

    return ecg_scaled, rr_scaled

def class_weights(labels):
    counts = np.bincount(labels, minlength=5)
    present = counts > 0
    weights = np.ones(5, dtype=np.float32)

    weights[present] = (
        len(labels) / (present.sum() * counts[present])
    )

    if not present.all():
        print("Lớp không có trong phần train:", np.array(CLASS_NAMES)[~present])

    return counts, torch.from_numpy(weights)

def make_loader(ecg, rr, labels, batch_size, shuffle=False, seed=42):
    dataset = TensorDataset(
        torch.from_numpy(np.ascontiguousarray(ecg, dtype=np.float32)),
        torch.from_numpy(np.ascontiguousarray(rr, dtype=np.float32)),
        torch.from_numpy(np.asarray(labels, dtype=np.int64)),
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
        generator=torch.Generator().manual_seed(seed),
    )

def train_epoch(model, loader, optimizer, weights):
    model.train()
    total_loss = 0.0
    total_weight = 0.0

    for ecg, rr, labels in loader:
        ecg = ecg.to(DEVICE)
        rr = rr.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(ecg, rr)
        numerator = F.cross_entropy(
            logits, labels, weight=weights, reduction="sum"
        )
        denominator = weights[labels].sum()
        loss = numerator / denominator

        if not torch.isfinite(loss):
            raise RuntimeError("Loss không hữu hạn.")

        loss.backward()
        optimizer.step()

        total_loss += numerator.detach().item()
        total_weight += denominator.detach().item()

    return total_loss / total_weight

def predict(model, loader):
    model.eval()
    truths = []
    probabilities = []
    loss_sum = 0.0

    with torch.inference_mode():
        for ecg, rr, labels in loader:
            logits = model(ecg.to(DEVICE), rr.to(DEVICE))
            loss_sum += F.cross_entropy(
                logits, labels.to(DEVICE), reduction="sum"
            ).item()
            truths.append(labels.numpy())
            probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())

    y_true = np.concatenate(truths)
    probs = np.concatenate(probabilities)

    if not np.isfinite(probs).all():
        raise RuntimeError("Xác suất có NaN hoặc Inf.")

    np.testing.assert_allclose(probs.sum(axis=1), 1.0, atol=1e-5)

    return y_true, probs, loss_sum / len(y_true)

def safe_divide(numerator, denominator):
    return float(numerator / denominator) if denominator > 0 else np.nan

def calculate_metrics(y_true, probabilities):
    y_pred = probabilities.argmax(axis=1)
    matrix = confusion_matrix(y_true, y_pred, labels=np.arange(5))
    details = []

    for index, name in enumerate(CLASS_NAMES):
        tp = int(matrix[index, index])
        fn = int(matrix[index].sum() - tp)
        fp = int(matrix[:, index].sum() - tp)
        tn = int(matrix.sum() - tp - fn - fp)

        details.append({
            "class": name,
            "support": int(matrix[index].sum()),
            "predicted_count": int(matrix[:, index].sum()),
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn,
            "Se": safe_divide(tp, tp + fn),
            "PPV": safe_divide(tp, tp + fp),
            "Sp": safe_divide(tn, tn + fp),
        })

    per_class = pd.DataFrame(details).set_index("class")

    metrics = {
        "se_S": float(per_class.loc["S", "Se"]),
        "ppv_S": float(per_class.loc["S", "PPV"]),
        "se_V": float(per_class.loc["V", "Se"]),
        "ppv_V": float(per_class.loc["V", "PPV"]),
        "f1_macro": float(f1_score(
            y_true, y_pred, labels=np.arange(5),
            average="macro", zero_division=0,
        )),
        "sp_macro": float(np.mean(per_class["Sp"].to_numpy())),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }

    return matrix, per_class.reset_index(), metrics

def fit_model(model, train_loader, val_loader, weights, configuration):
    model = model.to(DEVICE)
    weights = weights.to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=configuration["learning_rate"]
    )

    best_score = -np.inf
    best_state = None
    best_epoch = 0
    waiting = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, weights)
        labels, probabilities, val_loss = predict(model, val_loader)
        _, _, metrics = calculate_metrics(labels, probabilities)

        improved = metrics["f1_macro"] > best_score

        if improved:
            best_score = metrics["f1_macro"]
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            waiting = 0
        else:
            waiting += 1

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **metrics,
            "improved": improved,
        })

        print(
            f"Epoch {epoch:02d} | "
            f"Train loss {train_loss:.4f} | "
            f"Val loss {val_loss:.4f} | "
            f"F1 macro {metrics['f1_macro']:.4f}"
        )

        if waiting >= PATIENCE:
            break

    model.load_state_dict(best_state)

    return model, best_state, best_epoch, pd.DataFrame(history)

def prediction_table(data, indices, probabilities, variant, seed, **extra):
    frame = pd.DataFrame({
        "model_variant": variant,
        "seed": seed,
        "sample_index": indices,
        "record_id": data["record_id"][indices],
        "patient_id": data["patient_id"][indices],
        "r_peak_index": data["r_peak_index"][indices],
        "y_true": data["labels"][indices],
        "y_pred": probabilities.argmax(axis=1),
    })

    for index, name in enumerate(CLASS_NAMES):
        frame[f"p_{name}"] = probabilities[:, index]

    for key, value in extra.items():
        frame[key] = value

    return frame

for name, constructor in MODEL_CLASSES.items():
    model = constructor()
    parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(name, "| Số tham số:", parameters)
    print(model)

In [ ]:
splitter = GroupKFold(n_splits=N_FOLDS)

folds = list(splitter.split(
    ds1["ecg"],
    ds1["labels"],
    groups=ds1["record_id"],
))

fold_metadata = []
validation_counts = np.zeros(len(ds1["labels"]), dtype=np.int64)

for fold, (train_indices, val_indices) in enumerate(folds, start=1):
    train_patients = set(ds1["patient_id"][train_indices])
    val_patients = set(ds1["patient_id"][val_indices])
    intersection = train_patients & val_patients

    if intersection:
        raise ValueError(
            f"Fold {fold} trùng bệnh nhân: {sorted(intersection)}. "
            "Cần thống nhất lại cách nhóm bản ghi theo bệnh nhân."
        )

    validation_counts[val_indices] += 1

    fold_metadata.append({
        "fold": fold,
        "train_records": sorted(set(ds1["record_id"][train_indices])),
        "validation_records": sorted(set(ds1["record_id"][val_indices])),
        "train_patients": sorted(train_patients),
        "validation_patients": sorted(val_patients),
        "patient_intersection": sorted(intersection),
    })

    print(
        f"Fold {fold}: "
        f"train {len(train_indices)} | "
        f"validation {len(val_indices)} | "
        f"giao bệnh nhân {sorted(intersection)}"
    )

if not np.all(validation_counts == 1):
    raise RuntimeError("Mỗi nhịp phải xuất hiện đúng một lần trong validation.")

save_json(fold_metadata, OUT_DIR / "cv_split.json")

cv_rows = []

for configuration in CONFIGS:
    config_id = configuration["config_id"]

    for variant, constructor in MODEL_CLASSES.items():
        for seed in SEEDS:
            for fold, (train_indices, val_indices) in enumerate(folds, start=1):
                set_seed(seed)

                normalization = fit_normalization(
                    ds1["ecg"][train_indices],
                    ds1["rr"][train_indices],
                )

                train_ecg, train_rr = normalize(
                    ds1["ecg"][train_indices],
                    ds1["rr"][train_indices],
                    normalization,
                )

                val_ecg, val_rr = normalize(
                    ds1["ecg"][val_indices],
                    ds1["rr"][val_indices],
                    normalization,
                )

                counts, weights = class_weights(ds1["labels"][train_indices])

                train_loader = make_loader(
                    train_ecg, train_rr, ds1["labels"][train_indices],
                    configuration["batch_size"], shuffle=True, seed=seed,
                )

                val_loader = make_loader(
                    val_ecg, val_rr, ds1["labels"][val_indices],
                    configuration["batch_size"],
                )

                run_name = f"{config_id}_{variant}_seed{seed}_fold{fold}"

                print("\n", run_name)

                model, state, best_epoch, history = fit_model(
                    constructor(), train_loader, val_loader,
                    weights, configuration,
                )

                labels, probabilities, _ = predict(model, val_loader)
                _, _, metrics = calculate_metrics(labels, probabilities)

                checkpoint = {
                    "state_dict": state,
                    "model_variant": variant,
                    "seed": seed,
                    "fold": fold,
                    "configuration": configuration,
                    "best_epoch": best_epoch,
                    "normalization": normalization,
                    "class_counts": counts.tolist(),
                    "class_weights": weights.tolist(),
                    "class_names": CLASS_NAMES,
                    "rr_order": RR_ORDER,
                }

                torch.save(
                    checkpoint,
                    OUT_DIR / "checkpoints" / f"{run_name}.pt",
                )

                save_csv(history, OUT_DIR / "logs" / f"{run_name}.csv")

                np.savez_compressed(
                    OUT_DIR / "results" / f"{run_name}_validation.npz",
                    indices=val_indices,
                    probabilities=probabilities,
                )

                cv_rows.append({
                    "config_id": config_id,
                    "model_variant": variant,
                    "seed": seed,
                    "fold": fold,
                    "learning_rate": configuration["learning_rate"],
                    "batch_size": configuration["batch_size"],
                    "best_epoch": best_epoch,
                    "epochs_run": len(history),
                    **metrics,
                })

                save_csv(cv_rows, OUT_DIR / "results" / "cv_runs.csv")

                del model, train_loader, val_loader
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

cv_results = pd.DataFrame(cv_rows)

configuration_scores = (
    cv_results.groupby("config_id", as_index=False)
    .agg(mean_cv_f1=("f1_macro", "mean"))
    .sort_values(
        ["mean_cv_f1", "config_id"],
        ascending=[False, True],
    )
)

selected_id = configuration_scores.iloc[0]["config_id"]
selected_configuration = next(
    dict(item) for item in CONFIGS if item["config_id"] == selected_id
)

selected_runs = cv_results[cv_results["config_id"] == selected_id]

final_epochs = max(
    1,
    int(math.ceil(selected_runs["best_epoch"].median())),
)

selection = {
    "configuration": selected_configuration,
    "final_epochs": final_epochs,
    "selection_data": "DS1_only",
    "selection_rule": protocol["configuration_selection"],
    "final_epoch_rule": protocol["final_epoch_rule"],
}

save_csv(
    configuration_scores,
    OUT_DIR / "results" / "configuration_scores.csv",
)
save_json(selection, OUT_DIR / "selection.json")

print("\nBảng chọn cấu hình:")
print(configuration_scores.to_string(index=False))
print("\nCấu hình đã chốt:", selected_configuration)
print("Số epoch huấn luyện cuối:", final_epochs)

In [ ]:
def summarize_metrics(frame, group_column="model_variant"):
    rows = []

    for group_name, group in frame.groupby(group_column):
        for metric in METRICS:
            values = group[metric].to_numpy(dtype=float)
            valid = np.isfinite(values)
            complete = bool(valid.all())

            rows.append({
                group_column: group_name,
                "metric": metric,
                "mean": float(values.mean()) if complete else np.nan,
                "std": float(values.std(ddof=1))
                if complete and len(values) > 1 else np.nan,
                "valid_seeds": int(valid.sum()),
                "total_seeds": int(len(values)),
            })

    return pd.DataFrame(rows)

def comparison_table(summary):
    rows = []

    for variant, group in summary.groupby("model_variant"):
        row = {"model_variant": variant}

        for item in group.to_dict("records"):
            if np.isfinite(item["mean"]) and np.isfinite(item["std"]):
                text = f"{item['mean']:.4f} ± {item['std']:.4f}"
            else:
                text = (
                    f"Không xác định "
                    f"({item['valid_seeds']}/{item['total_seeds']} seed hợp lệ)"
                )

            row[item["metric"]] = text

        rows.append(row)

    return pd.DataFrame(rows)

def ablation_table(metrics_frame):
    table = metrics_frame.pivot(
        index="seed",
        columns="model_variant",
        values="f1_macro",
    ).reset_index()

    table = table.rename(columns={
        "baseline": "baseline_f1",
        "cnn_rr": "cnn_rr_f1",
    })

    table["delta_f1"] = table["cnn_rr_f1"] - table["baseline_f1"]
    table.columns.name = None

    return table

oof_prediction_parts = []
oof_metric_rows = []

for variant in MODEL_CLASSES:
    for seed in SEEDS:
        probabilities = np.full(
            (len(ds1["labels"]), 5), np.nan, dtype=np.float32
        )
        fold_ids = np.zeros(len(ds1["labels"]), dtype=np.int64)
        coverage = np.zeros(len(ds1["labels"]), dtype=np.int64)

        for fold in range(1, N_FOLDS + 1):
            run_name = f"{selected_id}_{variant}_seed{seed}_fold{fold}"
            path = OUT_DIR / "results" / f"{run_name}_validation.npz"

            with np.load(path, allow_pickle=False) as source:
                indices = source["indices"]
                probabilities[indices] = source["probabilities"]
                fold_ids[indices] = fold
                coverage[indices] += 1

        if not np.all(coverage == 1) or not np.isfinite(probabilities).all():
            raise RuntimeError("Dự đoán OOF thiếu hoặc trùng nhịp.")

        _, _, metrics = calculate_metrics(ds1["labels"], probabilities)

        oof_metric_rows.append({
            "model_variant": variant,
            "seed": seed,
            **metrics,
        })

        frame = prediction_table(
            ds1,
            np.arange(len(ds1["labels"])),
            probabilities,
            variant,
            seed,
            evaluation_set="DS1_OOF",
        )
        frame["fold"] = fold_ids
        oof_prediction_parts.append(frame)

oof_predictions = pd.concat(oof_prediction_parts, ignore_index=True)
oof_metrics = pd.DataFrame(oof_metric_rows)
oof_summary = summarize_metrics(oof_metrics)
oof_comparison = comparison_table(oof_summary)
oof_ablation = ablation_table(oof_metrics)

save_csv(oof_predictions, OUT_DIR / "results" / "oof_predictions.csv")
save_csv(oof_metrics, OUT_DIR / "results" / "oof_metrics.csv")
save_csv(oof_summary, OUT_DIR / "results" / "oof_summary.csv")
save_csv(oof_comparison, OUT_DIR / "results" / "oof_comparison.csv")
save_csv(oof_ablation, OUT_DIR / "results" / "oof_ablation.csv")

print(oof_comparison.to_string(index=False))
print("\nAblation DS1:")
print(oof_ablation.to_string(index=False))

In [ ]:
final_normalization = fit_normalization(ds1["ecg"], ds1["rr"])

full_ecg, full_rr = normalize(
    ds1["ecg"], ds1["rr"], final_normalization
)

final_counts, final_weights = class_weights(ds1["labels"])

save_json(final_normalization, OUT_DIR / "normalization.json")

final_checkpoint_paths = {}

for variant, constructor in MODEL_CLASSES.items():
    for seed in SEEDS:
        set_seed(seed)

        model = constructor().to(DEVICE)
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=selected_configuration["learning_rate"],
        )

        loader = make_loader(
            full_ecg, full_rr, ds1["labels"],
            selected_configuration["batch_size"],
            shuffle=True,
            seed=seed,
        )

        history = []
        weights = final_weights.to(DEVICE)

        for epoch in range(1, final_epochs + 1):
            loss = train_epoch(model, loader, optimizer, weights)
            history.append({"epoch": epoch, "train_loss": loss})

            print(
                f"{variant} | seed {seed} | "
                f"epoch {epoch}/{final_epochs} | loss {loss:.4f}"
            )

        checkpoint = {
            "state_dict": {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            },
            "model_variant": variant,
            "seed": seed,
            "configuration": selected_configuration,
            "final_epochs": final_epochs,
            "normalization": final_normalization,
            "class_counts": final_counts.tolist(),
            "class_weights": final_weights.tolist(),
            "class_names": CLASS_NAMES,
            "rr_order": RR_ORDER,
            "preprocess_config": preprocess_config,
            "training_scope": "full_DS1",
        }

        path = OUT_DIR / "checkpoints" / f"final_{variant}_seed{seed}.pt"
        torch.save(checkpoint, path)
        final_checkpoint_paths[(variant, seed)] = path

        save_csv(
            history,
            OUT_DIR / "logs" / f"final_{variant}_seed{seed}.csv",
        )

        del model, optimizer, loader
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Số mô hình cuối:", len(final_checkpoint_paths))

In [ ]:
ds2 = read_dataset(DATA_DIR / "ds2.npz", DS2_RECORDS)

record_intersection = sorted(
    set(ds1["record_id"]) & set(ds2["record_id"])
)
patient_intersection = sorted(
    set(ds1["patient_id"]) & set(ds2["patient_id"])
)

print("Giao bản ghi DS1/DS2:", record_intersection)
print("Giao bệnh nhân DS1/DS2:", patient_intersection)

if record_intersection or patient_intersection:
    raise ValueError("DS1 và DS2 đang bị rò rỉ dữ liệu.")

split_information = {
    "DS1": {
        "records": sorted(set(ds1["record_id"])),
        "patients": sorted(set(ds1["patient_id"])),
    },
    "DS2": {
        "records": sorted(set(ds2["record_id"])),
        "patients": sorted(set(ds2["patient_id"])),
    },
    "record_to_patient": {
        record: str(data["patient_id"][np.flatnonzero(data["record_id"] == record)[0]])
        for data in [ds1, ds2]
        for record in np.unique(data["record_id"])
    },
    "record_intersection": record_intersection,
    "patient_intersection": patient_intersection,
    "excluded_test_record": "202",
}

save_json(split_information, OUT_DIR / "split.json")

test_ecg, test_rr = normalize(
    ds2["ecg"], ds2["rr"], final_normalization
)

test_loader = make_loader(
    test_ecg, test_rr, ds2["labels"],
    selected_configuration["batch_size"],
)

test_rows = []
test_prediction_parts = []
test_probabilities = {}

for variant, constructor in MODEL_CLASSES.items():
    for seed in SEEDS:
        checkpoint = torch.load(
            final_checkpoint_paths[(variant, seed)],
            map_location="cpu",
            weights_only=True,
        )

        model = constructor().to(DEVICE)
        model.load_state_dict(checkpoint["state_dict"])

        labels, probabilities, loss = predict(model, test_loader)
        matrix, per_class, metrics = calculate_metrics(labels, probabilities)

        test_probabilities[(variant, seed)] = probabilities

        test_rows.append({
            "model_variant": variant,
            "seed": seed,
            "test_loss": loss,
            **metrics,
        })

        test_prediction_parts.append(prediction_table(
            ds2,
            np.arange(len(ds2["labels"])),
            probabilities,
            variant,
            seed,
            evaluation_set="DS2",
        ))

        matrix_frame = pd.DataFrame(
            matrix,
            index=CLASS_NAMES,
            columns=CLASS_NAMES,
        )
        matrix_frame.index.name = "true_class"

        matrix_frame.to_csv(
            OUT_DIR / "results" / f"confusion_{variant}_seed{seed}.csv",
            encoding="utf-8-sig",
        )

        save_csv(
            per_class,
            OUT_DIR / "results" / f"per_class_{variant}_seed{seed}.csv",
        )

        print(
            variant, "| seed", seed,
            "| F1 macro:", round(metrics["f1_macro"], 4),
        )

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

test_metrics = pd.DataFrame(test_rows)
test_predictions = pd.concat(test_prediction_parts, ignore_index=True)

test_summary = summarize_metrics(test_metrics)
test_comparison = comparison_table(test_summary)
test_ablation = ablation_table(test_metrics)

save_csv(test_metrics, OUT_DIR / "results" / "ds2_metrics.csv")
save_csv(test_predictions, OUT_DIR / "results" / "ds2_predictions.csv")
save_csv(test_summary, OUT_DIR / "results" / "ds2_summary.csv")
save_csv(test_comparison, OUT_DIR / "results" / "ds2_comparison.csv")
save_csv(test_ablation, OUT_DIR / "results" / "ds2_ablation.csv")

print("\nKết quả DS2:")
print(test_comparison.to_string(index=False))

print("\nAblation DS2:")
print(test_ablation.to_string(index=False))

print(
    "\nChênh lệch F1 macro:",
    f"{test_ablation['delta_f1'].mean():.4f}",
    "±",
    f"{test_ablation['delta_f1'].std(ddof=1):.4f}",
)

In [ ]:
checkpoint = torch.load(
    final_checkpoint_paths[(DEPLOYMENT_VARIANT, EXPORT_SEED)],
    map_location="cpu",
    weights_only=True,
)

saliency_model = CNNWithRR().to(DEVICE)
saliency_model.load_state_dict(checkpoint["state_dict"])
saliency_model.eval()

probabilities = test_probabilities[(DEPLOYMENT_VARIANT, EXPORT_SEED)]
predictions = probabilities.argmax(axis=1)

wrong_indices = np.flatnonzero(predictions != ds2["labels"])
correct_indices = np.flatnonzero(predictions == ds2["labels"])

priority = (
    np.isin(ds2["labels"][wrong_indices], [1, 2])
    | np.isin(predictions[wrong_indices], [1, 2])
)

wrong_indices = wrong_indices[np.argsort(~priority, kind="stable")][:10]
correct_indices = correct_indices[:2]
case_indices = np.concatenate([wrong_indices, correct_indices])

case_rows = []
ecg_gradients = []
rr_gradients = []

time_axis = (np.arange(234) - 90) / 360.0 * 1000.0

for index in case_indices:
    ecg_tensor = torch.tensor(
        test_ecg[index:index + 1],
        device=DEVICE,
        requires_grad=True,
    )
    rr_tensor = torch.tensor(
        test_rr[index:index + 1],
        device=DEVICE,
        requires_grad=True,
    )

    with torch.enable_grad():
        logits = saliency_model(ecg_tensor, rr_tensor)
        predicted_class = int(logits.argmax(dim=1).item())

        ecg_gradient, rr_gradient = torch.autograd.grad(
            logits[0, predicted_class],
            (ecg_tensor, rr_tensor),
        )

    ecg_gradient = ecg_gradient.detach().cpu().numpy()[0, 0]
    rr_gradient = rr_gradient.detach().cpu().numpy()[0]

    ecg_gradients.append(ecg_gradient)
    rr_gradients.append(rr_gradient)

    true_class = int(ds2["labels"][index])

    row = {
        "sample_index": int(index),
        "record_id": ds2["record_id"][index],
        "patient_id": ds2["patient_id"][index],
        "r_peak_index": int(ds2["r_peak_index"][index]),
        "y_true": true_class,
        "y_pred": predicted_class,
        "correct": bool(true_class == predicted_class),
    }

    for position, name in enumerate(CLASS_NAMES):
        row[f"p_{name}"] = float(probabilities[index, position])

    for position, name in enumerate(RR_ORDER):
        row[name] = float(ds2["rr"][index, position])

    case_rows.append(row)

    magnitude = np.abs(ecg_gradient)
    magnitude = magnitude / max(float(magnitude.max()), 1e-12)

    figure, axes = plt.subplots(
        3, 1, figsize=(10, 8), constrained_layout=True
    )

    axes[0].plot(time_axis, ds2["ecg"][index, 0])
    axes[0].axvline(0, color="gray", linestyle="--")
    axes[0].set_xlabel("Thời gian so với R (ms)")
    axes[0].set_ylabel("Biên độ ECG")
    axes[0].set_title(
        f"Bản ghi {ds2['record_id'][index]} | "
        f"Thật: {CLASS_NAMES[true_class]} | "
        f"Dự đoán: {CLASS_NAMES[predicted_class]} | "
        f"p = {probabilities[index, predicted_class]:.4f}"
    )

    axes[1].plot(time_axis, magnitude, color="darkred")
    axes[1].set_xlabel("Thời gian so với R (ms)")
    axes[1].set_ylabel("Saliency ECG tương đối")
    axes[1].set_ylim(0, 1.05)

    axes[2].bar(RR_ORDER, np.abs(rr_gradient), color="steelblue")
    axes[2].set_ylabel("|Gradient theo RR chuẩn hóa|")

    figure.savefig(
        OUT_DIR / "saliency" / f"sample_{int(index)}.png",
        dpi=180,
    )
    plt.show()
    plt.close(figure)

save_csv(case_rows, OUT_DIR / "results" / "saliency_cases.csv")

np.savez_compressed(
    OUT_DIR / "saliency" / "cases.npz",
    sample_index=case_indices,
    ecg_raw=ds2["ecg"][case_indices],
    rr_raw=ds2["rr"][case_indices],
    ecg_gradient=np.asarray(ecg_gradients),
    rr_gradient=np.asarray(rr_gradients),
)

print("Số trường hợp sai được xuất:", len(wrong_indices))
print("Số trường hợp đúng được xuất:", len(correct_indices))

del saliency_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
class ProbabilityModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, ecg, rr):
        return torch.softmax(self.model(ecg, rr), dim=1)


checkpoint = torch.load(
    final_checkpoint_paths[(DEPLOYMENT_VARIANT, EXPORT_SEED)],
    map_location="cpu",
    weights_only=True,
)

export_base = CNNWithRR()
export_base.load_state_dict(checkpoint["state_dict"])
export_base.eval()

export_model = ProbabilityModel(export_base).eval()

FP32_PATH = OUT_DIR / "exports" / "cnn_rr_fp32.onnx"
PREPARED_PATH = OUT_DIR / "exports" / "cnn_rr_prepared.onnx"
INT8_PATH = OUT_DIR / "exports" / "cnn_rr_int8.onnx"

batch_dimension = torch.export.Dim("batch", min=1)

torch.onnx.export(
    export_model,
    (
        torch.from_numpy(full_ecg[:4]),
        torch.from_numpy(full_rr[:4]),
    ),
    str(FP32_PATH),
    input_names=["ecg", "rr"],
    output_names=["probabilities"],
    dynamic_shapes={
        "ecg": {0: batch_dimension},
        "rr": {0: batch_dimension},
    },
    opset_version=18,
    dynamo=True,
    external_data=False,
)

onnx.checker.check_model(onnx.load(str(FP32_PATH)))

class ECGCalibrationReader(CalibrationDataReader):
    def __init__(self, ecg, rr, batch_size=128):
        self.ecg = np.ascontiguousarray(ecg, dtype=np.float32)
        self.rr = np.ascontiguousarray(rr, dtype=np.float32)
        self.batch_size = batch_size
        self.position = 0

    def get_next(self):
        if self.position >= len(self.ecg):
            return None

        end = min(self.position + self.batch_size, len(self.ecg))

        result = {
            "ecg": self.ecg[self.position:end],
            "rr": self.rr[self.position:end],
        }

        self.position = end
        return result

    def rewind(self):
        self.position = 0

calibration_rng = np.random.default_rng(EXPORT_SEED)
calibration_indices = calibration_rng.choice(
    len(ds1["labels"]),
    size=min(4096, len(ds1["labels"])),
    replace=False,
)

calibration_reader = ECGCalibrationReader(
    full_ecg[calibration_indices],
    full_rr[calibration_indices],
)

quant_pre_process(
    input_model=str(FP32_PATH),
    output_model_path=str(PREPARED_PATH),
    skip_optimization=True,
    skip_symbolic_shape=True,
)

quantize_static(
    model_input=str(PREPARED_PATH),
    model_output=str(INT8_PATH),
    calibration_data_reader=calibration_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
    calibrate_method=CalibrationMethod.MinMax,
    op_types_to_quantize=["Conv", "Gemm", "MatMul"],
)

int8_graph = onnx.load(str(INT8_PATH))
onnx.checker.check_model(int8_graph)

int8_tensor_count = sum(
    tensor.data_type in [onnx.TensorProto.INT8, onnx.TensorProto.UINT8]
    for tensor in int8_graph.graph.initializer
)

if int8_tensor_count == 0:
    raise RuntimeError("Không tìm thấy tensor INT8 trong mô hình.")

save_json({
    "source": "DS1_only",
    "seed": EXPORT_SEED,
    "sample_indices": calibration_indices.tolist(),
    "number_of_beats": len(calibration_indices),
    "quantization_format": "QDQ",
    "quantized_operations": ["Conv", "Gemm", "MatMul"],
    "input_dtype": "float32",
    "output_dtype": "float32",
}, OUT_DIR / "exports" / "calibration.json")

fp32_session = ort.InferenceSession(
    str(FP32_PATH),
    providers=["CPUExecutionProvider"],
)
int8_session = ort.InferenceSession(
    str(INT8_PATH),
    providers=["CPUExecutionProvider"],
)

batch_checks = []

for batch_size in [1, 4, 16, 64]:
    ecg_batch = np.ascontiguousarray(full_ecg[:batch_size])
    rr_batch = np.ascontiguousarray(full_rr[:batch_size])

    with torch.inference_mode():
        reference = export_model(
            torch.from_numpy(ecg_batch),
            torch.from_numpy(rr_batch),
        ).numpy()

    for version, session in [
        ("ONNX_FP32", fp32_session),
        ("ONNX_INT8", int8_session),
    ]:
        result = session.run(
            ["probabilities"],
            {"ecg": ecg_batch, "rr": rr_batch},
        )[0]

        if result.shape != (len(ecg_batch), 5):
            raise RuntimeError("Sai shape đầu ra ONNX.")

        if not np.isfinite(result).all():
            raise RuntimeError("Đầu ra ONNX không hữu hạn.")

        np.testing.assert_allclose(result.sum(axis=1), 1.0, atol=1e-5)

        if version == "ONNX_FP32":
            np.testing.assert_allclose(
                result, reference, rtol=1e-4, atol=1e-5
            )

        batch_checks.append({
            "version": version,
            "batch_size": len(ecg_batch),
            "max_probability_error": float(np.max(np.abs(result - reference))),
            "label_agreement": float(np.mean(
                result.argmax(axis=1) == reference.argmax(axis=1)
            )),
        })

save_csv(
    batch_checks,
    OUT_DIR / "results" / "onnx_batch_checks.csv",
)

print(pd.DataFrame(batch_checks).to_string(index=False))
print("Số tensor INT8:", int8_tensor_count)
print("FP32:", round(FP32_PATH.stat().st_size / 1024, 2), "KiB")
print("INT8:", round(INT8_PATH.stat().st_size / 1024, 2), "KiB")

In [ ]:
def predict_cpu_pytorch(model, ecg, rr, batch_size=256):
    parts = []

    with torch.inference_mode():
        for start in range(0, len(ecg), batch_size):
            end = start + batch_size
            parts.append(model(
                torch.from_numpy(ecg[start:end]),
                torch.from_numpy(rr[start:end]),
            ).numpy())

    return np.concatenate(parts)

def predict_onnx(session, ecg, rr, batch_size=256):
    parts = []

    for start in range(0, len(ecg), batch_size):
        end = start + batch_size

        result = session.run(
            ["probabilities"],
            {
                "ecg": np.ascontiguousarray(ecg[start:end], dtype=np.float32),
                "rr": np.ascontiguousarray(rr[start:end], dtype=np.float32),
            },
        )[0]

        if not np.isfinite(result).all():
            raise RuntimeError("Xác suất ONNX không hữu hạn.")

        np.testing.assert_allclose(result.sum(axis=1), 1.0, atol=1e-5)
        parts.append(result)

    return np.concatenate(parts)

conversion_probabilities = {
    "PyTorch_FP32": predict_cpu_pytorch(export_model, test_ecg, test_rr),
    "ONNX_FP32": predict_onnx(fp32_session, test_ecg, test_rr),
    "ONNX_INT8": predict_onnx(int8_session, test_ecg, test_rr),
}

reference = conversion_probabilities["PyTorch_FP32"]

np.testing.assert_allclose(
    conversion_probabilities["ONNX_FP32"],
    reference,
    rtol=1e-4,
    atol=1e-5,
)

conversion_rows = []
conversion_prediction_parts = []

for version, probabilities in conversion_probabilities.items():
    matrix, per_class, metrics = calculate_metrics(
        ds2["labels"], probabilities
    )

    conversion_rows.append({
        "version": version,
        **metrics,
        "max_probability_error": float(np.max(
            np.abs(probabilities - reference)
        )),
        "label_agreement": float(np.mean(
            probabilities.argmax(axis=1) == reference.argmax(axis=1)
        )),
        "changed_labels": int(np.sum(
            probabilities.argmax(axis=1) != reference.argmax(axis=1)
        )),
    })

    conversion_prediction_parts.append(prediction_table(
        ds2,
        np.arange(len(ds2["labels"])),
        probabilities,
        DEPLOYMENT_VARIANT,
        EXPORT_SEED,
        version=version,
        evaluation_set="DS2",
    ))

    save_csv(
        per_class,
        OUT_DIR / "results" / f"conversion_per_class_{version}.csv",
    )

conversion_metrics = pd.DataFrame(conversion_rows)

reference_metrics = conversion_metrics[
    conversion_metrics["version"] == "PyTorch_FP32"
].iloc[0]

delta_rows = []

for item in conversion_rows:
    if item["version"] == "PyTorch_FP32":
        continue

    delta_rows.append({
        "version": item["version"],
        **{
            f"delta_{metric}": item[metric] - reference_metrics[metric]
            for metric in METRICS
        },
    })

size_table = pd.DataFrame([
    {
        "version": "ONNX_FP32",
        "bytes": FP32_PATH.stat().st_size,
        "KiB": FP32_PATH.stat().st_size / 1024,
    },
    {
        "version": "ONNX_INT8",
        "bytes": INT8_PATH.stat().st_size,
        "KiB": INT8_PATH.stat().st_size / 1024,
    },
])

save_csv(conversion_metrics, OUT_DIR / "results" / "conversion_metrics.csv")
save_csv(delta_rows, OUT_DIR / "results" / "conversion_deltas.csv")
save_csv(
    pd.concat(conversion_prediction_parts, ignore_index=True),
    OUT_DIR / "results" / "conversion_predictions.csv",
)
save_csv(size_table, OUT_DIR / "results" / "model_sizes.csv")

np.savez_compressed(
    OUT_DIR / "exports" / "sample_io.npz",
    ecg_raw=ds2["ecg"][:4],
    rr_raw=ds2["rr"][:4],
    ecg_normalized=test_ecg[:4],
    rr_normalized=test_rr[:4],
    pytorch_probabilities=reference[:4],
    onnx_fp32_probabilities=conversion_probabilities["ONNX_FP32"][:4],
    onnx_int8_probabilities=conversion_probabilities["ONNX_INT8"][:4],
)

print(conversion_metrics.to_string(index=False))
print("\nChênh lệch chỉ số:")
print(pd.DataFrame(delta_rows).to_string(index=False))
print("\nKích thước:")
print(size_table.to_string(index=False))

In [ ]:
loaded_checkpoint = torch.load(
    final_checkpoint_paths[(DEPLOYMENT_VARIANT, EXPORT_SEED)],
    map_location="cpu",
    weights_only=True,
)

loaded_model = CNNWithRR()
loaded_model.load_state_dict(loaded_checkpoint["state_dict"])
loaded_model.eval()

with torch.inference_mode():
    loaded_probabilities = torch.softmax(
        loaded_model(
            torch.from_numpy(test_ecg[:4]),
            torch.from_numpy(test_rr[:4]),
        ),
        dim=1,
    ).numpy()

np.testing.assert_allclose(
    loaded_probabilities, reference[:4], rtol=1e-4, atol=1e-5
)

inference_configuration = {
    "model_version": RUN_ID,
    "model_variant": DEPLOYMENT_VARIANT,
    "seed": EXPORT_SEED,
    "deployment_model": "exports/cnn_rr_fp32.onnx",
    "experimental_int8_model": "exports/cnn_rr_int8.onnx",
    "input_names": ["ecg", "rr"],
    "ecg_shape": ["batch", 1, 234],
    "rr_shape": ["batch", 3],
    "input_dtype": "float32",
    "output_name": "probabilities",
    "output_shape": ["batch", 5],
    "class_names": CLASS_NAMES,
    "rr_order": RR_ORDER,
    "rr_unit_before_normalization": "seconds",
    "normalization_file": "normalization.json",
    "normalization_location": "outside_ONNX",
    "normalization_application": "exactly_once",
    "decision_rule": "argmax",
    "sampling_rate": 360,
    "preprocessing_file": "preprocess_config.json",
    "r_peak_source": "provided_annotations",
    "intended_use": "review_of_previously_acquired_Holter_recordings",
    "pytorch_checkpoint_output": "logits",
    "onnx_output": "probabilities",
}

save_json(
    inference_configuration,
    OUT_DIR / "inference_config.json",
)

packages = [
    "numpy", "pandas", "scikit-learn", "matplotlib",
    "torch", "onnx", "onnxruntime", "onnxscript",
]

versions = {
    name: importlib.metadata.version(name)
    for name in packages
}

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "packages": versions,
    "training_device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0)
    if torch.cuda.is_available() else None,
    "onnx_providers": ort.get_available_providers(),
}

save_json(environment, OUT_DIR / "environment.json")

requirements = "\n".join(
    f"{name}=={version.split('+')[0]}"
    for name, version in versions.items()
) + "\n"

(OUT_DIR / "requirements.txt").write_text(
    requirements, encoding="utf-8"
)

input_integrity = {
    name: {
        "md5": file_hash(DATA_DIR / name, "md5"),
        "sha256": file_hash(DATA_DIR / name, "sha256"),
    }
    for name in ["ds1.npz", "ds2.npz", "preprocess_config.json"]
}

save_json(input_integrity, OUT_DIR / "processed_input_hashes.json")

architecture = {
    "ECG_encoder": [
        "Conv1d(1,16,kernel=7,padding=3)",
        "ReLU",
        "MaxPool1d(2)",
        "Conv1d(16,32,kernel=5,padding=2)",
        "ReLU",
        "MaxPool1d(2)",
        "Conv1d(32,64,kernel=3,padding=1)",
        "ReLU",
        "AdaptiveAvgPool1d(1)",
        "Flatten",
    ],
    "baseline_classifier": [
        "Linear(64,32)", "ReLU", "Dropout(0.2)", "Linear(32,5)"
    ],
    "RR_branch": [
        "Linear(3,16)", "ReLU", "Linear(16,16)", "ReLU"
    ],
    "CNN_RR_classifier": [
        "Linear(80,32)", "ReLU", "Dropout(0.2)", "Linear(32,5)"
    ],
    "parameter_counts": {
        name: sum(
            parameter.numel()
            for parameter in constructor().parameters()
            if parameter.requires_grad
        )
        for name, constructor in MODEL_CLASSES.items()
    },
}

save_json(architecture, OUT_DIR / "architecture.json")

readme = """
Phần mô hình và đánh giá của thành viên 2

Mã nguồn đầy đủ nằm trong notebook Kaggle đi kèm.
Lưu và tải notebook .ipynb sau khi chạy thành công toàn bộ các cell.

Dữ liệu đầu vào:
ds1.npz, ds2.npz và preprocess_config.json do thành viên 1 cung cấp.
Các nhịp được lọc và trích RR nhưng chưa chuẩn hóa bằng thống kê học từ dữ liệu.
Sửa DATA_DIR trong notebook rồi chạy lần lượt từ đầu đến cuối.

Quy trình:
Chọn cấu hình bằng GroupKFold trong DS1.
Hai mô hình dùng cùng tập cấu hình, seed, giới hạn epoch và patience.
Huấn luyện cuối trên toàn bộ DS1 theo cấu hình đã chốt.
Đánh giá DS2 sau khi chốt cấu hình.
Không dùng DS2 để chọn seed, đổi kiến trúc hoặc dò siêu tham số.

Checkpoint:
Các checkpoint phục vụ suy luận, không chứa trạng thái optimizer để tiếp tục train.
Khởi tạo lớp mô hình trong notebook rồi load state_dict.
Đầu ra PyTorch là logits; dùng softmax để lấy xác suất.

Bàn giao cho thành viên 1:
results/ds2_predictions.csv để bootstrap theo bệnh nhân/bản ghi.
results/saliency_cases.csv và thư mục saliency để phân tích lỗi.
Khoảng tin cậy bootstrap phải được bổ sung vào báo cáo kết quả.
processed_input_hashes.json là mã băm dữ liệu đã xử lý,
không thay thế MD5 của các tệp WFDB gốc do thành viên 1 quản lý.

Bàn giao cho thành viên 3:
exports/cnn_rr_fp32.onnx là phiên bản triển khai đã định trước.
exports/cnn_rr_int8.onnx phục vụ đối chiếu lượng tử hóa.
Chuẩn hóa đầu vào ngoài ONNX bằng normalization.json, đúng một lần.
Dùng đúng preprocess_config.json, thứ tự RR và thứ tự lớp.
exports/sample_io.npz chứa đầu vào và đầu ra tham chiếu.
API phải cho kết quả tương ứng với chương trình đánh giá.
Thành viên 3 thực hiện Docker, AI agent và đo độ trễ, RAM,
thời gian xử lý tổng lượng tín hiệu tương đương một giờ.

Chỉ số:
Ô trống trong CSV là chỉ số không xác định.
Độ lệch chuẩn qua seed không thay thế khoảng tin cậy bootstrap 95%.
OOF trong DS1 đã tham gia chọn checkpoint và cấu hình.
Kết quả kiểm tra độc lập cuối cùng được báo cáo trên DS2.

Mã nguồn và dữ liệu:
Nộp notebook cùng các tệp cấu hình, kết quả và mô hình cần thiết.
Không đưa dữ liệu ECG gốc, dữ liệu nhịp hoặc khóa truy cập vào kho Git.
Chỉ dùng sample_io trong phạm vi bàn giao nội bộ được phép.
Đối với kho mã nộp, nên tạo dữ liệu kiểm thử tổng hợp thay cho mẫu ECG thật.
Khai báo việc dùng công cụ AI theo Phụ lục B của hướng dẫn học phần.
""".strip()

(OUT_DIR / "README.txt").write_text(readme, encoding="utf-8")

output_manifest = {
    str(path.relative_to(OUT_DIR)): file_hash(path)
    for path in sorted(OUT_DIR.rglob("*"))
    if path.is_file() and path.name != "sha256_manifest.json"
}

save_json(output_manifest, OUT_DIR / "sha256_manifest.json")

zip_path = shutil.make_archive(
    str(OUT_DIR),
    "zip",
    root_dir=str(OUT_DIR),
)

print("Kiểm tra tải lại checkpoint: thành công.")
print("Số mô hình cuối:", len(final_checkpoint_paths))
print("Phiên bản triển khai:", DEPLOYMENT_VARIANT, EXPORT_SEED, DEPLOYMENT_FORMAT)
print("Thư mục kết quả:", OUT_DIR)
print("Gói bàn giao:", zip_path)
print("Kích thước ZIP:", round(Path(zip_path).stat().st_size / 1024, 2), "KiB")
print("Cần lưu và tải thêm notebook .ipynb chứa toàn bộ mã nguồn.")

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display

display(FileLink(Path(zip_path).name))

In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")

prediction_files = sorted(
    input_root.rglob("ds2_predictions.csv")
)

zip_files = sorted(
    path
    for path in input_root.rglob("*")
    if path.is_file()
    and path.suffix.lower() == ".zip"
    and "tv2" in path.name.lower()
)

if prediction_files:
    for path in prediction_files:
        print("Tìm thấy kết quả:", path)
elif zip_files:
    for path in zip_files:
        print("Tìm thấy ZIP:", path)
else:
    print("Chưa tìm thấy kết quả TV2. Kiểm tra đã thêm dataset vào Input chưa.")

In [ ]:
from pathlib import Path
import shutil

source_dir = Path(
    "/kaggle/input/datasets/quanghuy10/tv2-results"
)

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")

required_files = [
    "results/ds2_predictions.csv",
    "results/ds2_metrics.csv",
    "results/conversion_metrics.csv",
    "results/model_sizes.csv",
    "logs/lr_001_baseline_seed42_fold1.csv",
    "logs/lr_001_cnn_rr_seed42_fold1.csv",
]

missing = [
    name for name in required_files
    if not (source_dir / name).is_file()
]

if missing:
    raise FileNotFoundError(
        f"Dataset thiếu các tệp: {missing}"
    )

shutil.copytree(
    source_dir,
    OUT_DIR,
    dirs_exist_ok=True,
)

print("Đã khôi phục kết quả tại:", OUT_DIR)
print("Số tệp CSV:", len(list(OUT_DIR.rglob("*.csv"))))
print("Số hình saliency:", len(list((OUT_DIR / "saliency").glob("*.png"))))
print("Sẵn sàng tạo biểu đồ, không cần train lại.")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

models = {
    "baseline": "CNN cơ sở",
    "cnn_rr": "CNN + RR",
}

for variant, model_name in models.items():
    history_path = (
        OUT_DIR / "logs"
        / f"lr_001_{variant}_seed42_fold1.csv"
    )

    history = pd.read_csv(history_path).sort_values("epoch")

    required = {"epoch", "train_loss", "val_loss", "f1_macro"}

    if not required.issubset(history.columns):
        raise ValueError(
            f"{history_path.name} thiếu cột: "
            f"{sorted(required - set(history.columns))}"
        )

    best_row = history.loc[history["f1_macro"].idxmax()]
    best_epoch = int(best_row["epoch"])
    best_f1 = float(best_row["f1_macro"])

    figure, axes = plt.subplots(
        1, 2,
        figsize=(12, 5),
        constrained_layout=True,
    )

    axes[0].plot(
        history["epoch"],
        history["train_loss"],
        marker="o",
        color="#2563EB",
        label="Train loss có trọng số",
    )

    axes[0].plot(
        history["epoch"],
        history["val_loss"],
        marker="s",
        color="#EA580C",
        label="Validation loss không trọng số",
    )

    axes[0].set_title("Diễn biến loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.25)

    axes[1].plot(
        history["epoch"],
        history["f1_macro"],
        marker="o",
        color="#15803D",
        label="F1 macro validation",
    )

    axes[1].axvline(
        best_epoch,
        color="#DC2626",
        linestyle="--",
        label=f"Checkpoint tốt nhất: epoch {best_epoch}",
    )

    axes[1].scatter(
        [best_epoch],
        [best_f1],
        color="#DC2626",
        s=65,
        zorder=5,
    )

    axes[1].set_title("Diễn biến F1 macro")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("F1 macro validation")
    axes[1].set_ylim(0, 1)
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.25)

    for axis in axes:
        axis.set_xticks(history["epoch"].astype(int))
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

    figure.suptitle(
        f"{model_name} — DS1 — seed 42 — fold 1 — learning rate 0.001",
        fontsize=13,
        fontweight="bold",
    )

    filename = f"training_{variant}_seed42_fold1"

    figure.savefig(
        FIG_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        FIG_DIR / f"{filename}.pdf",
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    print(
        f"{model_name}: epoch tốt nhất = {best_epoch}, "
        f"F1 macro validation = {best_f1:.4f}"
    )

print("\nĐã lưu 2 hình PNG và 2 bản PDF tại:")
print(FIG_DIR)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

class_names = ["N", "S", "V", "F", "Q"]
model_names = {
    "baseline": "CNN cơ sở",
    "cnn_rr": "CNN + RR",
}
seeds = [42, 123, 2026]

saved_count = 0

for variant, model_name in model_names.items():
    for seed in seeds:
        matrix_path = (
            OUT_DIR / "results"
            / f"confusion_{variant}_seed{seed}.csv"
        )

        table = pd.read_csv(matrix_path)

        matrix = (
            table.set_index("true_class")
            .loc[class_names, class_names]
            .to_numpy(dtype=np.int64)
        )

        if matrix.shape != (5, 5) or np.any(matrix < 0):
            raise ValueError(
                f"Ma trận không hợp lệ: {matrix_path.name}"
            )

        support = matrix.sum(axis=1, keepdims=True)

        normalized = np.divide(
            matrix.astype(float),
            support,
            out=np.zeros((5, 5), dtype=float),
            where=support > 0,
        )

        figure, axes = plt.subplots(
            1, 2,
            figsize=(13, 5.5),
            constrained_layout=True,
        )

        for axis, values, title, is_normalized in [
            (axes[0], matrix, "Số nhịp", False),
            (axes[1], normalized, "Tỷ lệ theo nhãn thật", True),
        ]:
            image = axis.imshow(
                values,
                cmap="Blues",
                vmin=0,
                vmax=1 if is_normalized else None,
            )

            threshold = (
                0.5 if is_normalized
                else float(matrix.max()) / 2
            )

            for row in range(5):
                for column in range(5):
                    label = (
                        f"{values[row, column]:.1%}"
                        if is_normalized
                        else f"{int(values[row, column]):,}"
                    )

                    axis.text(
                        column,
                        row,
                        label,
                        ha="center",
                        va="center",
                        fontsize=10,
                        color=(
                            "white"
                            if values[row, column] > threshold
                            else "black"
                        ),
                    )

            axis.set_xticks(range(5), class_names)
            axis.set_yticks(range(5), class_names)
            axis.set_xlabel("Nhãn dự đoán")
            axis.set_ylabel("Nhãn thật")
            axis.set_title(title)

            figure.colorbar(
                image,
                ax=axis,
                label="Tỷ lệ" if is_normalized else "Số nhịp",
                shrink=0.85,
            )

        figure.suptitle(
            f"{model_name} — Ma trận nhầm lẫn DS2 — seed {seed}",
            fontsize=14,
            fontweight="bold",
        )

        filename = f"confusion_ds2_{variant}_seed{seed}"

        figure.savefig(
            FIG_DIR / f"{filename}.png",
            dpi=300,
            bbox_inches="tight",
        )

        figure.savefig(
            FIG_DIR / f"{filename}.pdf",
            bbox_inches="tight",
        )

        if seed == 42:
            plt.show()

        plt.close(figure)
        saved_count += 1

        print(
            f"{model_name} | seed {seed} | "
            f"{int(matrix.sum()):,} nhịp | Đã lưu hình"
        )

print(
    f"\nĐã lưu {saved_count} hình PNG "
    f"và {saved_count} bản PDF."
)
print("Thư mục:", FIG_DIR)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

metrics = pd.read_csv(
    OUT_DIR / "results" / "ds2_metrics.csv"
)

model_order = ["baseline", "cnn_rr"]
model_names = {
    "baseline": "CNN cơ sở",
    "cnn_rr": "CNN + RR",
}
colors = {
    "baseline": "#2563EB",
    "cnn_rr": "#EA580C",
}

metric_columns = [
    "f1_macro", "sp_macro", "accuracy",
    "se_S", "ppv_S", "se_V", "ppv_V",
]

required_columns = {
    "model_variant", "seed", *metric_columns
}

if not required_columns.issubset(metrics.columns):
    raise ValueError(
        f"Thiếu cột: {sorted(required_columns - set(metrics.columns))}"
    )

metrics = metrics[
    metrics["model_variant"].isin(model_order)
].copy()

for variant in model_order:
    rows = metrics[metrics["model_variant"] == variant]

    if (
        len(rows) != 3
        or sorted(rows["seed"].astype(int).tolist()) != [42, 123, 2026]
    ):
        raise ValueError(
            f"{variant} phải có đúng ba seed 42, 123, 2026."
        )

    if not np.isfinite(
        rows[metric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f"{variant} có chỉ số không xác định."
        )

means = (
    metrics.groupby("model_variant")[metric_columns]
    .mean()
    .reindex(model_order)
)

stds = (
    metrics.groupby("model_variant")[metric_columns]
    .std(ddof=1)
    .reindex(model_order)
)

plot_groups = [
    (
        ["f1_macro", "sp_macro", "accuracy"],
        ["F1 macro", "Sp macro", "Accuracy"],
        "So sánh chỉ số tổng hợp trên DS2",
        "ds2_overall_comparison",
    ),
    (
        ["se_S", "ppv_S", "se_V", "ppv_V"],
        ["Se S", "PPV S", "Se V", "PPV V"],
        "So sánh chỉ số lớp S và V trên DS2",
        "ds2_sv_comparison",
    ),
]

for columns, labels, title, filename in plot_groups:
    positions = np.arange(len(columns))
    width = 0.36

    figure, axis = plt.subplots(
        figsize=(11, 6),
        constrained_layout=True,
    )

    for model_index, variant in enumerate(model_order):
        values = means.loc[variant, columns].to_numpy(dtype=float)
        errors = stds.loc[variant, columns].to_numpy(dtype=float)
        bar_positions = positions + (model_index - 0.5) * width

        axis.bar(
            bar_positions,
            values,
            width=width,
            yerr=errors,
            capsize=5,
            color=colors[variant],
            label=model_names[variant],
            error_kw={"elinewidth": 1.3},
        )

        for position, value, error in zip(
            bar_positions, values, errors
        ):
            axis.text(
                position,
                value + error + 0.018,
                f"{value:.3f}",
                ha="center",
                va="bottom",
                fontsize=10,
            )

    axis.set_xticks(positions, labels)
    axis.set_ylim(0, 1.10)
    axis.set_yticks(np.arange(0, 1.01, 0.1))
    axis.set_ylabel("Giá trị chỉ số")
    axis.set_title(
        title + "\nTrung bình ± SD qua ba seed: 42, 123, 2026",
        fontsize=13,
        pad=16,
    )
    axis.legend(
        loc="upper left",
        bbox_to_anchor=(0, 1.01),
        ncol=2,
        frameon=False,
    )
    axis.grid(axis="y", alpha=0.25)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

    figure.savefig(
        FIG_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        FIG_DIR / f"{filename}.pdf",
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

print("F1 macro trên DS2:")

for variant in model_order:
    print(
        f"{model_names[variant]}: "
        f"{means.loc[variant, 'f1_macro']:.4f} ± "
        f"{stds.loc[variant, 'f1_macro']:.4f}"
    )

print("\nĐã lưu thêm 2 hình PNG và 2 bản PDF.")
print("Thư mục:", FIG_DIR)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ablation = pd.read_csv(
    OUT_DIR / "results" / "ds2_ablation.csv"
).sort_values("seed")

required = {
    "seed", "baseline_f1", "cnn_rr_f1", "delta_f1"
}

if not required.issubset(ablation.columns):
    raise ValueError(
        f"Thiếu cột: {sorted(required - set(ablation.columns))}"
    )

if (
    len(ablation) != 3
    or ablation["seed"].astype(int).tolist() != [42, 123, 2026]
):
    raise ValueError("Cần đúng ba seed 42, 123, 2026.")

values = ablation[
    ["baseline_f1", "cnn_rr_f1", "delta_f1"]
].to_numpy(dtype=float)

if not np.isfinite(values).all():
    raise ValueError("Bảng ablation có giá trị không hữu hạn.")

np.testing.assert_allclose(
    ablation["delta_f1"],
    ablation["cnn_rr_f1"] - ablation["baseline_f1"],
    rtol=1e-6,
    atol=1e-8,
)

delta_mean = ablation["delta_f1"].mean()
delta_std = ablation["delta_f1"].std(ddof=1)

figure, axes = plt.subplots(
    1, 2,
    figsize=(12, 5.5),
    constrained_layout=True,
)

seed_colors = ["#2563EB", "#EA580C", "#15803D"]

for color, row in zip(seed_colors, ablation.itertuples(index=False)):
    axes[0].plot(
        [0, 1],
        [row.baseline_f1, row.cnn_rr_f1],
        marker="o",
        markersize=7,
        linewidth=2,
        color=color,
        label=f"Seed {int(row.seed)}",
    )

axes[0].set_xticks([0, 1], ["CNN cơ sở", "CNN + RR"])
axes[0].set_xlim(-0.2, 1.2)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("F1 macro trên DS2")
axes[0].set_title("So sánh theo cùng seed")
axes[0].legend()

positions = np.arange(len(ablation))

bars = axes[1].bar(
    positions,
    ablation["delta_f1"],
    color=seed_colors,
    width=0.6,
)

for bar, value in zip(bars, ablation["delta_f1"]):
    axes[1].annotate(
        f"{value:+.4f}",
        xy=(bar.get_x() + bar.get_width() / 2, value),
        xytext=(0, 6 if value >= 0 else -6),
        textcoords="offset points",
        ha="center",
        va="bottom" if value >= 0 else "top",
    )

axes[1].axhline(0, color="black", linewidth=0.9)

axes[1].axhline(
    delta_mean,
    color="#7C3AED",
    linestyle="--",
    linewidth=1.5,
    label=f"Trung bình ΔF1 = {delta_mean:.4f}",
)

axes[1].set_xticks(
    positions,
    ablation["seed"].astype(int).astype(str),
)
axes[1].set_xlabel("Seed")
axes[1].set_ylabel("ΔF1 macro = CNN + RR − CNN cơ sở")
axes[1].set_title(
    f"Cải thiện khi thêm RR\n"
    f"Trung bình ± SD: {delta_mean:.4f} ± {delta_std:.4f}"
)

lower = min(0.0, float(ablation["delta_f1"].min()))
upper = max(0.0, float(ablation["delta_f1"].max()))
margin = max((upper - lower) * 0.25, 0.03)
axes[1].set_ylim(lower - margin, upper + margin)
axes[1].legend(loc="upper left", fontsize=9)

for axis in axes:
    axis.grid(axis="y", alpha=0.25)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

figure.suptitle(
    "Ablation nhánh RR trên DS2",
    fontsize=14,
    fontweight="bold",
)

figure.savefig(
    FIG_DIR / "ds2_rr_ablation.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    FIG_DIR / "ds2_rr_ablation.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print(ablation.to_string(index=False))
print(
    f"\nChênh lệch F1 macro trung bình ± SD: "
    f"{delta_mean:.4f} ± {delta_std:.4f}"
)
print("Đã lưu thêm 1 hình PNG và 1 bản PDF.")
print("Thư mục:", FIG_DIR)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

conversion = pd.read_csv(
    OUT_DIR / "results" / "conversion_metrics.csv"
)

sizes = pd.read_csv(
    OUT_DIR / "results" / "model_sizes.csv"
)

with (OUT_DIR / "inference_config.json").open(
    encoding="utf-8"
) as file:
    inference_config = json.load(file)

export_seed = int(inference_config["seed"])

versions = [
    "PyTorch_FP32",
    "ONNX_FP32",
    "ONNX_INT8",
]

required_metrics = {
    "version", "f1_macro", "accuracy",
    "se_S", "ppv_S", "se_V", "ppv_V",
    "changed_labels", "label_agreement",
}

if not required_metrics.issubset(conversion.columns):
    raise ValueError(
        f"Thiếu cột: "
        f"{sorted(required_metrics - set(conversion.columns))}"
    )

if conversion["version"].duplicated().any():
    raise ValueError("Bảng chuyển đổi có phiên bản trùng.")

if sizes["version"].duplicated().any():
    raise ValueError("Bảng kích thước có phiên bản trùng.")

conversion = conversion.set_index("version").loc[versions]

size_versions = ["ONNX_FP32", "ONNX_INT8"]
sizes = sizes.set_index("version").loc[size_versions].copy()
sizes["KiB"] = sizes["bytes"] / 1024

metric_columns = [
    "f1_macro", "accuracy",
    "se_S", "ppv_S", "se_V", "ppv_V",
]

if not np.isfinite(
    conversion[metric_columns].to_numpy(dtype=float)
).all():
    raise ValueError("Bảng chuyển đổi có chỉ số không xác định.")

if not np.isfinite(sizes["KiB"]).all() or (sizes["KiB"] <= 0).any():
    raise ValueError("Kích thước mô hình không hợp lệ.")

version_labels = [
    "PyTorch\nFP32",
    "ONNX\nFP32",
    "ONNX\nINT8",
]

colors = ["#64748B", "#2563EB", "#EA580C"]

figure, axes = plt.subplots(
    1, 2,
    figsize=(12, 5.5),
    constrained_layout=True,
)

size_bars = axes[0].bar(
    ["ONNX FP32", "ONNX INT8"],
    sizes["KiB"].to_numpy(),
    color=[colors[1], colors[2]],
    width=0.55,
)

axes[0].bar_label(size_bars, fmt="%.2f", padding=5)
axes[0].set_ylabel("Kích thước tệp (KiB)")
axes[0].set_title("Kích thước mô hình")
axes[0].set_ylim(0, float(sizes["KiB"].max()) * 1.25)

f1_bars = axes[1].bar(
    version_labels,
    conversion["f1_macro"].to_numpy(),
    color=colors,
    width=0.55,
)

axes[1].bar_label(f1_bars, fmt="%.4f", padding=5)
axes[1].set_ylabel("F1 macro trên DS2")
axes[1].set_title("Chất lượng phân loại trước và sau chuyển đổi")
axes[1].set_ylim(0, 1)

for axis in axes:
    axis.grid(axis="y", alpha=0.25)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

figure.suptitle(
    f"CNN + RR — checkpoint seed {export_seed}",
    fontsize=14,
    fontweight="bold",
)

figure.savefig(
    FIG_DIR / "conversion_size_f1.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    FIG_DIR / "conversion_size_f1.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

columns = ["se_S", "ppv_S", "se_V", "ppv_V"]
labels = ["Se S", "PPV S", "Se V", "PPV V"]
positions = np.arange(len(columns))
width = 0.25

figure, axis = plt.subplots(
    figsize=(12, 6),
    constrained_layout=True,
)

for version_index, version in enumerate(versions):
    values = conversion.loc[
        version, columns
    ].to_numpy(dtype=float)

    bar_positions = positions + (version_index - 1) * width

    bars = axis.bar(
        bar_positions,
        values,
        width=width,
        color=colors[version_index],
        label=version.replace("_", " "),
    )

    axis.bar_label(
        bars,
        fmt="%.3f",
        padding=4,
        fontsize=9,
    )

axis.set_xticks(positions, labels)
axis.set_ylim(0, 1.12)
axis.set_yticks(np.arange(0, 1.01, 0.1))
axis.set_ylabel("Giá trị chỉ số trên DS2")
axis.set_title(
    f"Se và PPV lớp S, V trước và sau chuyển đổi\n"
    f"CNN + RR — checkpoint seed {export_seed}",
    fontsize=13,
    pad=16,
)
axis.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.01),
    ncol=3,
    frameon=False,
)
axis.grid(axis="y", alpha=0.25)
axis.set_axisbelow(True)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)

figure.savefig(
    FIG_DIR / "conversion_sv_comparison.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    FIG_DIR / "conversion_sv_comparison.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

size_reduction = (
    1 - sizes.loc["ONNX_INT8", "bytes"]
    / sizes.loc["ONNX_FP32", "bytes"]
) * 100

f1_delta = (
    conversion.loc["ONNX_INT8", "f1_macro"]
    - conversion.loc["PyTorch_FP32", "f1_macro"]
)

print("So sánh chuyển đổi trên DS2:")
print(
    conversion[
        metric_columns + ["label_agreement", "changed_labels"]
    ].to_string()
)

print(f"\nINT8 giảm kích thước tệp: {size_reduction:.2f}%")
print(f"Chênh lệch F1 macro INT8 so với PyTorch: {f1_delta:+.6f}")
print("Đã lưu thêm 2 hình PNG và 2 bản PDF.")
print("Thư mục:", FIG_DIR)

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

with (OUT_DIR / "architecture.json").open(
    encoding="utf-8"
) as file:
    architecture = json.load(file)

parameter_counts = architecture["parameter_counts"]

plt.rcParams["font.family"] = "DejaVu Sans"

def draw_box(axis, x, y, width, height, text, color):
    box = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.04",
        linewidth=1.3,
        edgecolor="#334155",
        facecolor=color,
    )
    axis.add_patch(box)
    axis.text(
        x + width / 2,
        y + height / 2,
        text,
        ha="center",
        va="center",
        fontsize=11,
        linespacing=1.4,
    )

def draw_arrow(axis, start, end):
    axis.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops={
            "arrowstyle": "->",
            "linewidth": 1.6,
            "color": "#334155",
            "shrinkA": 3,
            "shrinkB": 3,
        },
    )

def save_architecture(figure, filename):
    figure.savefig(
        FIG_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIG_DIR / f"{filename}.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(figure)

encoder_text = (
    "Bộ mã hóa ECG\n"
    "Conv1D 1 → 16, kernel 7 + ReLU\n"
    "MaxPool1D 2\n"
    "Conv1D 16 → 32, kernel 5 + ReLU\n"
    "MaxPool1D 2\n"
    "Conv1D 32 → 64, kernel 3 + ReLU\n"
    "AdaptiveAvgPool1D 1 + Flatten\n"
    "Đặc trưng ECG: [B, 64]"
)

figure, axis = plt.subplots(figsize=(8, 10))
axis.set_xlim(0, 10)
axis.set_ylim(0, 10)
axis.axis("off")

draw_box(
    axis, 2.5, 8.6, 5, 0.9,
    "ECG đã chuẩn hóa\n[B, 1, 234]",
    "#DBEAFE",
)

draw_box(
    axis, 1.5, 5.2, 7, 2.9,
    encoder_text,
    "#EFF6FF",
)

draw_box(
    axis, 2.5, 2.9, 5, 1.6,
    "Bộ phân loại\n"
    "Linear 64 → 32 + ReLU\n"
    "Dropout 0.2\n"
    "Linear 32 → 5: logits [B, 5]",
    "#FFEDD5",
)

draw_box(
    axis, 2.5, 1.1, 5, 1.1,
    "Softmax khi suy luận\n"
    "Xác suất [B, 5]\n"
    "Thứ tự lớp: N, S, V, F, Q",
    "#DCFCE7",
)

draw_arrow(axis, (5, 8.6), (5, 8.1))
draw_arrow(axis, (5, 5.2), (5, 4.5))
draw_arrow(axis, (5, 2.9), (5, 2.2))

axis.set_title(
    f"CNN cơ sở\n"
    f"{parameter_counts['baseline']:,} tham số học được",
    fontsize=15,
    fontweight="bold",
    pad=15,
)

axis.text(
    5, 0.35,
    "B: số nhịp trong batch. Loss huấn luyện nhận logits.",
    ha="center",
    fontsize=10,
)

figure.tight_layout()
save_architecture(figure, "architecture_baseline")

figure, axis = plt.subplots(figsize=(9, 11))
axis.set_xlim(0, 10)
axis.set_ylim(0, 11)
axis.axis("off")

draw_box(
    axis, 0.25, 9.6, 4.3, 0.9,
    "ECG đã chuẩn hóa\n[B, 1, 234]",
    "#DBEAFE",
)

draw_box(
    axis, 5.45, 9.6, 4.3, 0.9,
    "RR đã chuẩn hóa\n[B, 3]",
    "#DBEAFE",
)

draw_box(
    axis, 0.25, 5.9, 4.3, 3.2,
    encoder_text,
    "#EFF6FF",
)

draw_box(
    axis, 5.45, 6.6, 4.3, 1.8,
    "Nhánh RR\n"
    "Linear 3 → 16 + ReLU\n"
    "Linear 16 → 16 + ReLU\n"
    "Đặc trưng RR: [B, 16]",
    "#EFF6FF",
)

draw_box(
    axis, 2.5, 4.4, 5, 0.9,
    "Ghép đặc trưng ECG và RR\n"
    "64 + 16 = 80 chiều: [B, 80]",
    "#F3E8FF",
)

draw_box(
    axis, 2.5, 2.3, 5, 1.5,
    "Bộ phân loại\n"
    "Linear 80 → 32 + ReLU\n"
    "Dropout 0.2\n"
    "Linear 32 → 5: logits [B, 5]",
    "#FFEDD5",
)

draw_box(
    axis, 2.5, 0.7, 5, 1,
    "Softmax khi suy luận\n"
    "Xác suất [B, 5]\n"
    "Thứ tự lớp: N, S, V, F, Q",
    "#DCFCE7",
)

draw_arrow(axis, (2.4, 9.6), (2.4, 9.1))
draw_arrow(axis, (7.6, 9.6), (7.6, 8.4))
draw_arrow(axis, (2.4, 5.9), (3.6, 5.3))
draw_arrow(axis, (7.6, 6.6), (6.4, 5.3))
draw_arrow(axis, (5, 4.4), (5, 3.8))
draw_arrow(axis, (5, 2.3), (5, 1.7))

axis.set_title(
    f"CNN + RR\n"
    f"{parameter_counts['cnn_rr']:,} tham số học được",
    fontsize=15,
    fontweight="bold",
    pad=15,
)

axis.text(
    5, 0.15,
    "B: số nhịp trong batch. Loss huấn luyện nhận logits.",
    ha="center",
    fontsize=10,
)

figure.tight_layout()
save_architecture(figure, "architecture_cnn_rr")

print("CNN cơ sở:", parameter_counts["baseline"], "tham số")
print("CNN + RR:", parameter_counts["cnn_rr"], "tham số")
print("Đã lưu thêm 2 sơ đồ PNG và 2 bản PDF.")
print("Thư mục:", FIG_DIR)

In [ ]:
from pathlib import Path
import json
import hashlib
import zipfile
from IPython.display import FileLink, display

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
SALIENCY_DIR = OUT_DIR / "saliency"

png_files = sorted(FIG_DIR.glob("*.png"))
pdf_files = sorted(FIG_DIR.glob("*.pdf"))
saliency_files = sorted(SALIENCY_DIR.glob("*.png"))

expected_figures = [
    "training_baseline_seed42_fold1",
    "training_cnn_rr_seed42_fold1",
    *[
        f"confusion_ds2_{variant}_seed{seed}"
        for variant in ["baseline", "cnn_rr"]
        for seed in [42, 123, 2026]
    ],
    "ds2_overall_comparison",
    "ds2_sv_comparison",
    "ds2_rr_ablation",
    "conversion_size_f1",
    "conversion_sv_comparison",
    "architecture_baseline",
    "architecture_cnn_rr",
]

missing = [
    f"{name}.{extension}"
    for name in expected_figures
    for extension in ["png", "pdf"]
    if not (FIG_DIR / f"{name}.{extension}").is_file()
]

if missing:
    raise FileNotFoundError(
        f"Chưa có đủ hình để đóng gói: {missing}"
    )

if len(saliency_files) != 12:
    raise ValueError(
        f"Cần 12 ảnh saliency, hiện tìm thấy {len(saliency_files)}."
    )

table_names = [
    "saliency_cases.csv",
    "ds2_metrics.csv",
    "ds2_summary.csv",
    "ds2_ablation.csv",
    "conversion_metrics.csv",
    "conversion_deltas.csv",
    "model_sizes.csv",
]

table_files = [
    OUT_DIR / "results" / name
    for name in table_names
]

missing_tables = [
    path.name for path in table_files
    if not path.is_file()
]

if missing_tables:
    raise FileNotFoundError(
        f"Thiếu bảng kết quả: {missing_tables}"
    )

package_files = (
    png_files
    + pdf_files
    + saliency_files
    + table_files
)

zip_path = Path("/kaggle/working/TV2_REPORT_FIGURES.zip")
manifest = {}

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in package_files:
        archive_name = path.relative_to(OUT_DIR).as_posix()
        archive.write(path, arcname=archive_name)

        manifest[archive_name] = hashlib.sha256(
            path.read_bytes()
        ).hexdigest()

    archive.writestr(
        "sha256_manifest.json",
        json.dumps(
            manifest,
            ensure_ascii=False,
            indent=2,
        ),
    )

with zipfile.ZipFile(zip_path, "r") as archive:
    bad_file = archive.testzip()
    if bad_file is not None:
        raise RuntimeError(
            f"Tệp lỗi trong ZIP: {bad_file}"
        )

print("Đóng gói thành công.")
print("Hình báo cáo PNG:", len(png_files))
print("Hình báo cáo PDF:", len(pdf_files))
print("Ảnh saliency:", len(saliency_files))
print("Bảng kết quả:", len(table_files))
print("Tên ZIP:", zip_path.name)
print("Đường dẫn:", zip_path)
print(
    "Kích thước:",
    round(zip_path.stat().st_size / (1024 ** 2), 2),
    "MiB",
)

display(FileLink("TV2_REPORT_FIGURES.zip"))